In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Basic Models with just phase interaction
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Define required columns based on usage in the notebook
required_columns = [
    'isBounty',
    'userFEIsBounty',
    'userId',
    'year',
    'timestamp',
    'timeSinceFirstActivityDays',
    'logtimeSinceFirstActivityDays',
    'userFeLogTimeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'userFeLogNumHelpProvidedAT',
    'userFeLogNumQuestionsAskedAT',
]

df = pd.read_parquet('../data/study_datasets/user_answers_bounty_processed.parquet',
                    columns=required_columns)

len_before = len(df)
# Convert timestamp to datetime if it's not already
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Filter to remove all rows where timestamp is before or at January 27, 2009 = introduction of bounty stem
cutoff_date = pd.to_datetime('2009-01-27')
df = df[df['timestamp'] > cutoff_date]

len_after = len(df)
print(f"Removed {len_before - len_after} rows. Final length: {len_after}")
df.info()

In [ ]:
df["userId"].nunique()

# Load Data

In [3]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Basic Models with just phase interaction
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Define required columns based on usage in the notebook
required_columns = [
    'isBounty',
    'userFEIsBounty',
    'userId',
    'year',
    'timestamp',
    'timeSinceFirstActivityDays',
    'logtimeSinceFirstActivityDays',
    'userFeLogTimeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'userFeLogNumHelpProvidedAT',
    'userFeLogNumQuestionsAskedAT',
]

df = pd.read_parquet('../data/study_datasets/user_answers_bounty_processed.parquet',
                    columns=required_columns)

# Convert timestamp to datetime if it's not already
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Filter to remove all rows where timestamp is before or at January 27, 2009 = introduction of bounty stem
cutoff_date = pd.to_datetime('2009-01-27')
df = df[df['timestamp'] > cutoff_date]

# Recalculate userFEIsBounty after filtering
df['userFEIsBounty'] = df.groupby("userId")["isBounty"].transform(lambda x: x - x.mean())

print(f"Data shape after filtering: {df.shape}")
print(f"Date range after filtering: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.info()

Data shape after filtering: (32627382, 14)
Date range after filtering: 2009-01-27 00:00:02.727000 to 2025-03-31 23:59:12.307000
<class 'pandas.core.frame.DataFrame'>
Index: 32627382 entries, 0 to 32856257
Data columns (total 14 columns):
 #   Column                               Dtype         
---  ------                               -----         
 0   isBounty                             int64         
 1   userFEIsBounty                       float64       
 2   userId                               int64         
 3   year                                 int32         
 4   timestamp                            datetime64[ns]
 5   timeSinceFirstActivityDays           float64       
 6   logtimeSinceFirstActivityDays        float64       
 7   userFeLogTimeSinceFirstActivityDays  float32       
 8   numHelpProvidedAT                    int64         
 9   numQuestionsAskedAT                  int64         
 10  logNumHelpProvidedAT                 float64       
 11  logNumQuestionsA

# Get Descriptives

In [ ]:
# Define columns for descriptive statistics
descriptive_columns = [
    'isBounty',
    'timeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'logtimeSinceFirstActivityDays'
]

def calculate_descriptives(data, columns):
    """Calculate descriptive statistics (μ, σ, min, max) for specified columns"""
    stats_dict = {}

    for col in columns:
        if col in data.columns:
            # Handle non-numeric columns by skipping them or converting
            if data[col].dtype in ['object', 'category']:
                continue

            stats_dict[col] = {
                'μ': data[col].mean(),
                'σ': data[col].std(),
                'min': data[col].min(),
                'max': data[col].max()
            }
        else:
            print(f"Warning: Column '{col}' not found in data")
    return pd.DataFrame(stats_dict).T

# Calculate overall descriptive statistics
print("=== DESCRIPTIVE STATISTICS ===")
overall_stats = calculate_descriptives(df, descriptive_columns)
print(overall_stats.round(4))

# 1. Main Bounty Effect

In [ ]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
import gc
import os

sys.modules['stargazer.translators.statsmodels'].pd = pd

df["logNumHelpProvidedAT2"] = df["logNumHelpProvidedAT"] ** 2
df["logNumQuestionsAskedAT2"] = df["logNumQuestionsAskedAT"] ** 2

# Non-FE Models
formula1 = "isBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logtimeSinceFirstActivityDays + C(year)"

# FE Models
formula2 = "userFEIsBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logtimeSinceFirstActivityDays + C(year)"

model_names = ["1", "2"]
formulas = [formula1, formula2]

def fit_model(formula, model_name, data):
    """Fit a single model, save LaTeX results, and clean memory"""
    print(f"Fitting Model {model_name}...")
    print(f"Formula: {formula}")

    try:
        # Clear memory before fitting
        gc.collect()

        # Fit the model
        model = smf.ols(formula=formula, data=data).fit(
            cov_type='cluster',
            cov_kwds={'groups': data['userId']}
        )

        print(f"✓ Model {model_name} fitted successfully")

        # Create individual Stargazer table for this model
        stargazer = Stargazer([model])
        stargazer.title(f"Model {model_name}: Effect of Receiving Answers on Providing Help")
        stargazer.significant_digits(3)
        stargazer.show_degrees_of_freedom(False)
        stargazer.show_model_numbers(True)

        # Generate LaTeX code
        latex_output = stargazer.render_latex()
        print(latex_output)

        # Clear the model from memory
        del model
        gc.collect()

        return True

    except MemoryError as e:
        print(f"✗ Memory error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False
    except Exception as e:
        print(f"✗ Error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False

successful_models = []
failed_models = []

for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"Processing {i+1}/{len(formulas)}")
    fit_model(formula, name, df)

In [ ]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
import gc
import os

sys.modules['stargazer.translators.statsmodels'].pd = pd

df["logNumHelpProvidedAT2"] = df["logNumHelpProvidedAT"] ** 2
df["logNumQuestionsAskedAT2"] = df["logNumQuestionsAskedAT"] ** 2

# Non-FE Models
formula1 = "isBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logNumHelpProvidedAT2 + logtimeSinceFirstActivityDays + C(year)"

# FE Models
formula2 = "userFEIsBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logNumHelpProvidedAT2 + logtimeSinceFirstActivityDays + C(year)"

model_names = ["1", "2"]
formulas = [formula1, formula2]

def fit_model(formula, model_name, data):
    """Fit a single model, save LaTeX results, and clean memory"""
    print(f"Fitting Model {model_name}...")
    print(f"Formula: {formula}")

    try:
        # Clear memory before fitting
        gc.collect()

        # Fit the model
        model = smf.ols(formula=formula, data=data).fit(
            cov_type='cluster',
            cov_kwds={'groups': data['userId']}
        )

        print(f"✓ Model {model_name} fitted successfully")

        # Create individual Stargazer table for this model
        stargazer = Stargazer([model])
        stargazer.title(f"Model {model_name}: Effect of Receiving Answers on Providing Help")
        stargazer.significant_digits(3)
        stargazer.show_degrees_of_freedom(False)
        stargazer.show_model_numbers(True)

        # Generate LaTeX code
        latex_output = stargazer.render_latex()
        print(latex_output)

        # Clear the model from memory
        del model
        gc.collect()

        return True

    except MemoryError as e:
        print(f"✗ Memory error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False
    except Exception as e:
        print(f"✗ Error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False

successful_models = []
failed_models = []

for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"Processing {i+1}/{len(formulas)}")
    fit_model(formula, name, df)

In [5]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
import gc
import os

df["logNumHelpProvidedAT2"] = df["logNumHelpProvidedAT"] ** 2
df["logNumQuestionsAskedAT2"] = df["logNumQuestionsAskedAT"] ** 2

formula2 = "userFEIsBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logNumHelpProvidedAT2 + logtimeSinceFirstActivityDays + C(year)"

print("Model 2 with increased decimal precision...")
print(f"Formula: {formula2}")

model2 = smf.ols(formula=formula2, data=df).fit(
            cov_type='cluster',
            cov_kwds={'groups': df['userId']}
)

# Assuming model2 is already fitted and stored somewhere, create Stargazer with more precision
stargazer = Stargazer([model2])  # Replace model2 with your actual fitted model variable
stargazer.title("Model 2: Effect of Receiving Answers on Providing Help (High Precision)")
stargazer.significant_digits(10)  # Increased from 3 to 6 for more decimal places
stargazer.show_degrees_of_freedom(False)
stargazer.show_model_numbers(True)

# Alternative: You can also try using decimal_char and precision methods
# stargazer.custom_columns(['Coefficient'], [1])  # Optional: customize column names

# Generate LaTeX code with higher precision
latex_output = stargazer.render_latex()
print(latex_output)

Model 2 with increased decimal precision...
Formula: userFEIsBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logNumHelpProvidedAT2 + logtimeSinceFirstActivityDays + C(year)


AttributeError: 'Stargazer' object has no attribute 'digits'

In [7]:
stargazer = Stargazer([model2])  # Replace model2 with your actual fitted model variable
stargazer.title("Model 2: Effect of Receiving Answers on Providing Help (High Precision)")
stargazer.significant_digits(8)  # Increased from 3 to 6 for more decimal places
stargazer.show_degrees_of_freedom(False)
stargazer.show_model_numbers(True)

# Alternative: You can also try using decimal_char and precision methods
# stargazer.custom_columns(['Coefficient'], [1])  # Optional: customize column names

# Generate LaTeX code with higher precision
latex_output = stargazer.render_latex()
print(latex_output)

\begin{table}[!htbp] \centering
  \caption{Model 2: Effect of Receiving Answers on Providing Help (High Precision)}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{1}{c}{\textit{Dependent variable: userFEIsBounty}} \
\cr \cline{2-2}
\\[-1.8ex] & (1) \\
\hline \\[-1.8ex]
 C(year)[T.2010] & -0.00088096$^{***}$ \\
& (0.00020365) \\
 C(year)[T.2011] & -0.00145666$^{***}$ \\
& (0.00016853) \\
 C(year)[T.2012] & -0.00125886$^{***}$ \\
& (0.00016226) \\
 C(year)[T.2013] & -0.00125213$^{***}$ \\
& (0.00016009) \\
 C(year)[T.2014] & -0.00121465$^{***}$ \\
& (0.00016067) \\
 C(year)[T.2015] & -0.00138734$^{***}$ \\
& (0.00016269) \\
 C(year)[T.2016] & -0.00135418$^{***}$ \\
& (0.00016091) \\
 C(year)[T.2017] & -0.00144258$^{***}$ \\
& (0.00018780) \\
 C(year)[T.2018] & -0.00145809$^{***}$ \\
& (0.00018215) \\
 C(year)[T.2019] & -0.00158361$^{***}$ \\
& (0.00016391) \\
 C(year)[T.2020] & -0.00033394$^{**}$ \\
& (0.00016906) \\
 C(year)[T.2021] & -0.000510